# 00 — Setup & Smoke Test

This notebook validates the end-to-end Colab workflow. No kernel logic is tested here — just the infrastructure:
- Clone / pull the repo
- Install dependencies
- Verify GPU access
- Verify the `kernels` package is importable
- Verify Triton is available

In [ ]:
# ── Cell 1: Clone or pull the repo ──────────────────────────────────────────
import os

REPO_URL = "https://github.com/YOUR_USERNAME/triton-kernels.git"  # update this
REPO_DIR = "/content/triton-kernels"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    !git -C {REPO_DIR} pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
!bash scripts/setup_colab.sh

In [ ]:
# ── Cell 3: Verify GPU ───────────────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "No GPU found! Enable GPU in Runtime > Change runtime type."

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}")
print(f"Memory: {props.total_memory / 1e9:.1f} GB")
print(f"Compute capability: {props.major}.{props.minor}")
print(f"SM count: {props.multi_processor_count}")

In [ ]:
# ── Cell 4: Verify package imports ──────────────────────────────────────────
import kernels
from kernels import elementwise, reductions, scanning, matmul, fft, attention

print("kernels package: OK")
print(f"Submodules: {[m for m in dir(kernels) if not m.startswith('_')]}")

In [ ]:
# ── Cell 5: Verify Triton ────────────────────────────────────────────────────
import triton
import triton.language as tl

print(f"Triton version: {triton.__version__}")

# Quick smoke test: a trivial kernel that adds a scalar to a vector
@triton.jit
def _smoke_kernel(x_ptr, out_ptr, scalar, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + scalar, mask=mask)

n = 1024
x = torch.ones(n, device='cuda')
out = torch.empty_like(x)
grid = lambda meta: (triton.cdiv(n, meta['BLOCK']),)
_smoke_kernel[grid](x, out, 2.0, n, BLOCK=128)

torch.testing.assert_close(out, torch.full((n,), 3.0, device='cuda'))
print("Triton smoke test: PASSED")

In [ ]:
# ── Cell 6: Summary ──────────────────────────────────────────────────────────
import sys
import numpy as np

print("=" * 50)
print("Environment Summary")
print("=" * 50)
print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Triton:  {triton.__version__}")
print(f"NumPy:   {np.__version__}")
print(f"GPU:     {torch.cuda.get_device_name(0)}")
print("=" * 50)
print("All checks passed! Ready to run kernel notebooks.")